# Tiêu chí 1 · Bám prompt

Từ thư mục gốc HELM-demo, nén ảnh và JSON thành hai tệp riêng:

```bash
python3 -m zipfile -c outputs/benchmark/images.zip outputs/benchmark/alignment/images
python3 -m zipfile -c outputs/benchmark/json.zip outputs/benchmark/alignment/json
```

Chọn `INPUT_SOURCE` trong ô đầu:

- `drive`: Tải hai ZIP lên `MyDrive/HEIM/allignment/` hoặc sửa đường dẫn trong ô đầu. ZIP kết quả được lưu cạnh ZIP ảnh.
- `colab`: Chọn ZIP ảnh khi chạy ô đầu và ZIP JSON khi chạy ô kiểm tra đầu vào. Sau khi chấm, Colab tải ZIP kết quả xuống.

Notebook ghép ảnh với JSON cùng tên và dùng caption gốc cho hai thước đo Alignment trong Bảng 3, mục 5 của HEIM: CLIPScore và CLIPScore với Multilingual CLIP. Cả hai dùng cùng các cặp ảnh–caption. Nếu có điểm chấm tay, đặt `OBSERVATIONS_JSONL` tới tệp tương ứng trong runtime hoặc trên Drive đã gắn. Có thể dùng trực tiếp ZIP chung cho tiêu chí 1, 2 và 8 theo [hướng dẫn Colab](../README.md).

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

INPUT_SOURCE = "drive"  # @param ["drive", "colab"]
CRITERION_ID = "1"
CRITERION_NAME = "alignment"
IMAGE_RUN = "vita-ask-uat-alignment-coco2014-2000prompts-val2014"
OBSERVATIONS_JSONL = None  # Ví dụ: Path("/content/drive/MyDrive/HEIM/allignment/observations.jsonl")

if INPUT_SOURCE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/HEIM/allignment")
    INPUT_ZIP = DRIVE_DIR / "images.zip"
    METADATA_ZIP = DRIVE_DIR / "json.zip"
elif INPUT_SOURCE == "colab":
    from google.colab import files

    UPLOAD_DIR = Path(mkdtemp(prefix="helm-upload-"))
    INPUT_ZIP = UPLOAD_DIR / "alignment-images.zip"
    METADATA_ZIP = UPLOAD_DIR / "alignment-json.zip"
    print("Chọn ZIP chứa thư mục images/.")
    files.upload_file(str(INPUT_ZIP))
else:
    raise ValueError(f"Nguồn đầu vào không hợp lệ: {INPUT_SOURCE}")

if not INPUT_ZIP.is_file():
    raise FileNotFoundError(f"Không tìm thấy ZIP ảnh: {INPUT_ZIP}")

## Giải nén và kiểm tra đầu vào

In [ ]:
import json
import re
import sqlite3
import stat
from contextlib import closing
from pathlib import PurePosixPath
from zipfile import ZipFile

WORK_DIR = Path(mkdtemp(prefix="helm-demo-"))
with ZipFile(INPUT_ZIP) as archive:
    entries = archive.infolist()
    image_zip = "evaluation.json" not in archive.namelist()
    members = set()
    for entry in entries:
        member = PurePosixPath(entry.filename)
        if member.is_absolute() or ".." in member.parts or stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError(f"Đường dẫn ZIP không hợp lệ: {entry.filename}")
        if image_zip:
            directory = entry.filename == "images/"
            image = not entry.is_dir() and re.fullmatch(r"images/id-[1-9][0-9]*\.png", entry.filename)
            if member in members or not (directory or image):
                raise ValueError(f"Mục ZIP ảnh không hợp lệ: {entry.filename}")
            members.add(member)
    archive.extractall(WORK_DIR)

if image_zip:
    run = IMAGE_RUN
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,79}", run):
        raise ValueError("Tên IMAGE_RUN không hợp lệ.")
    images = sorted((WORK_DIR / "images").glob("id-*.png"))
    if not images:
        raise ValueError("ZIP không có ảnh images/id-<id>.png.")
    if INPUT_SOURCE == "colab" and not METADATA_ZIP.is_file():
        print("Chọn ZIP chứa thư mục json/.")
        files.upload_file(str(METADATA_ZIP))
    if not METADATA_ZIP.is_file():
        raise FileNotFoundError(f"Không tìm thấy ZIP JSON: {METADATA_ZIP}")

    samples = {}
    with ZipFile(METADATA_ZIP) as archive:
        metadata = {}
        for entry in archive.infolist():
            if entry.filename == "json/" and entry.is_dir():
                continue
            if (entry.is_dir() or stat.S_ISLNK(entry.external_attr >> 16)
                    or not re.fullmatch(r"(?:json/)?id-[1-9][0-9]*\.json", entry.filename)):
                raise ValueError(f"Mục ZIP JSON không hợp lệ: {entry.filename}")
            stem = PurePosixPath(entry.filename).stem
            if stem in metadata:
                raise ValueError(f"ZIP có nhiều JSON cho ảnh {stem}.")
            metadata[stem] = entry

        missing = [image.name for image in images if image.stem not in metadata]
        if missing:
            raise ValueError(f"Thiếu JSON cho {len(missing)} ảnh, ví dụ: {', '.join(missing[:5])}")

        generation_suffix = (
            "\n\nGenerate exactly one image from the description above "
            "at a resolution of 512x512 pixels."
        )
        for image in images:
            entry = metadata[image.stem]
            with archive.open(entry) as source:
                sample = json.load(source)
            if (not isinstance(sample, dict) or sample.get("image_filename") != image.name
                    or not all(isinstance(sample.get(key), str) and sample[key].strip()
                               for key in ("prompt", "dataset", "source_id"))):
                raise ValueError(f"JSON thiếu dữ liệu hoặc không khớp với ảnh: {entry.filename}")
            caption = sample["prompt"].removesuffix(generation_suffix)
            if not caption.strip():
                raise ValueError(f"Thiếu caption trong {entry.filename}.")
            samples[image.name] = {**sample, "prompt": caption}
    image_count = len(images)
else:
    manifest = json.loads((WORK_DIR / "evaluation.json").read_text(encoding="utf-8"))
    run = manifest["run"]
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,79}", run):
        raise ValueError("Tên lượt trong evaluation.json không hợp lệ.")
    if CRITERION_ID not in manifest["criteria"]:
        raise ValueError(f"ZIP không có dữ liệu cho tiêu chí {CRITERION_ID} ({CRITERION_NAME}).")
    db_relative = Path("results/benchmark") / run / "requests.sqlite3"
    db_path = WORK_DIR / db_relative
    if "database" in manifest and manifest["database"] != db_relative.as_posix():
        raise ValueError("Đường dẫn SQLite trong evaluation.json không khớp với lượt chạy.")
    if not db_path.is_file():
        legacy_db = WORK_DIR / "outputs/benchmark/runs" / run / "requests.sqlite3"
        if not legacy_db.is_file():
            raise FileNotFoundError(f"ZIP thiếu SQLite: {db_relative}")
        db_path.parent.mkdir(parents=True, exist_ok=True)
        legacy_db.replace(db_path)

    image_counts = manifest.get("image_counts")
    if image_counts is None:
        with closing(sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)) as connection:
            image_counts = dict(connection.execute("""
                SELECT criterion, COUNT(*) FROM results
                WHERE json_extract(record, '$.status') = 'success'
                GROUP BY criterion
            """))
    image_count = image_counts.get(CRITERION_NAME, 0)
print(f"Lượt chạy {run} · {CRITERION_NAME}: {image_count:,} ảnh")

In [ ]:
import subprocess
import sys

if image_count:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "Pillow", "crfm-helm==0.5.16", "torchmetrics==0.11.4", "transformers==4.57.6",
            "sentence-transformers==5.1.2", "scipy", "tqdm",
        ],
        check=True,
    )

## Đánh giá, hiện kết quả và xuất ZIP

Notebook tách ảnh ghép 1024×512 thành hai ảnh 512×512 theo thứ tự trái rồi phải. Ảnh ghép 1024×1024 của lượt cũ được tách theo thứ tự hàng. Mỗi ảnh con được chấm với cùng caption gốc. CLIPScore dùng ViT-L/14 với đầu vào 512×512 và nội suy mã hóa vị trí. Multilingual CLIPScore dùng bước tiền xử lý mặc định của mô hình đa ngôn ngữ. Thanh tách ảnh hiển thị số ảnh đầu vào đã xử lý và số ảnh con đã tạo. Mỗi chỉ số có thanh đếm số ảnh con đã chấm.

Trong ZIP kết quả, `clip_scores.csv` ghi CLIPScore ở độ phân giải đầu vào 512×512 cho từng ảnh con. Cột `tile_index` cho biết vị trí trong ảnh ghép, bắt đầu từ 1. `summary.json` ghi trung bình các điểm theo prompt và khoảng tin cậy 95% theo phân phối t. Mỗi prompt được tính một lần sau khi lấy trung bình điểm các ảnh con của prompt đó.

In [ ]:
import csv
import subprocess
import sys
from collections import Counter, defaultdict
from contextlib import ExitStack
from html import escape
from math import sqrt
from statistics import mean, stdev
from tempfile import NamedTemporaryFile, TemporaryDirectory
from zipfile import ZIP_DEFLATED

from IPython.display import HTML, display
from tqdm.auto import tqdm


CLIP_MODEL_NAME = "openai/clip-vit-large-patch14"
CLIP_INPUT_SIZE = 512


def clip_scores_512(rows):
    import torch
    from PIL import Image, ImageOps
    from torch.nn.functional import cosine_similarity
    from transformers import CLIPModel, CLIPProcessor

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
    model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device).eval()
    batch_size = 4 if device.type == "cuda" else 1
    input_size = (CLIP_INPUT_SIZE, CLIP_INPUT_SIZE)

    with tqdm(total=len(rows), desc="CLIPScore 512×512", unit="ảnh") as progress:
        for start in range(0, len(rows), batch_size):
            batch = rows[start:start + batch_size]
            with ExitStack() as stack:
                images = []
                for row in batch:
                    with Image.open(row["image_path"]) as image:
                        rgb = stack.enter_context(image.convert("RGB"))
                        if rgb.size != input_size:
                            rgb = stack.enter_context(ImageOps.fit(
                                rgb, input_size, method=Image.Resampling.BICUBIC,
                            ))
                        images.append(rgb)
                pixel_values = processor.image_processor(
                    images=images,
                    do_resize=False,
                    do_center_crop=False,
                    return_tensors="pt",
                )["pixel_values"].to(device)
            text = processor.tokenizer(
                [row["sample"]["prompt"] for row in batch],
                padding=True,
                truncation=True,
                max_length=model.config.text_config.max_position_embeddings,
                return_tensors="pt",
            ).to(device)
            with torch.inference_mode():
                output = model(
                    **text, pixel_values=pixel_values, interpolate_pos_encoding=True,
                )
                scores = 100 * cosine_similarity(output.image_embeds, output.text_embeds).clamp(0, 1)
                scored = scores.cpu().tolist()
            progress.update(len(scored))
            yield from scored


# Same scoring as ImageMetrics.multilingual_clip_scores in the project evaluator.
def multilingual_clip_scores(rows):
    from PIL import Image
    from sentence_transformers import SentenceTransformer

    image_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32")
    text_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1")
    batch_size = 16
    with tqdm(total=len(rows), desc="Multilingual CLIPScore", unit="ảnh") as progress:
        for start in range(0, len(rows), batch_size):
            batch = rows[start:start + batch_size]
            with ExitStack() as stack:
                images = []
                for row in batch:
                    with Image.open(row["image_path"]) as image:
                        images.append(stack.enter_context(image.convert("RGB")))
                image_vectors = image_model.encode(images, normalize_embeddings=True, show_progress_bar=False)
            prompts = [row["sample"]["prompt"] for row in batch]
            text_vectors = text_model.encode(prompts, normalize_embeddings=True, show_progress_bar=False)
            cosine = (image_vectors * text_vectors).sum(axis=1).clip(min=0, max=1)
            progress.update(len(cosine))
            yield from (float(score) for score in 100 * cosine)



def prompt_key(sample):
    return sample["dataset"], str(sample["source_id"])


def split_tiles(rows, tile_dir):
    from PIL import Image

    tile_rows = []
    with tqdm(total=len(rows), desc="Tách ảnh ghép", unit="ảnh đầu vào") as progress:
        for row in rows:
            image_path = Path(row["image_path"])
            with Image.open(image_path) as image:
                width, height = image.size
                if not row["composite"]:
                    tile_rows.append({**row, "image_filename": image_path.name, "tile_index": 1})
                else:
                    if (width, height) not in {(1024, 512), (1024, 1024)}:
                        raise ValueError(f"Kích thước ảnh ghép không hợp lệ: {image_path.name} ({width}×{height}).")
                    for index in range((width // 512) * (height // 512)):
                        tile_row, tile_column = divmod(index, 2)
                        left, top = tile_column * 512, tile_row * 512
                        tile_path = tile_dir / f"{len(tile_rows) + 1}.png"
                        with image.crop((left, top, left + 512, top + 512)) as tile:
                            tile.save(tile_path)
                        tile_rows.append({
                            "image_path": tile_path,
                            "image_filename": image_path.name,
                            "tile_index": index + 1,
                            "sample": row["sample"],
                        })
            progress.set_postfix_str(f"{len(tile_rows):,} ảnh con", refresh=False)
            progress.update(1)
    return tile_rows


def prompt_level_metric(tile_rows, scores):
    by_prompt = defaultdict(list)
    for row, score in zip(tile_rows, scores, strict=True):
        by_prompt[prompt_key(row["sample"])].append(score)

    prompt_means = [mean(values) for values in by_prompt.values()]
    n = len(prompt_means)
    value = mean(prompt_means) if prompt_means else None
    repetitions = {len(values) for values in by_prompt.values()}
    metric = {
        "value": value,
        "n": n,
        "expected": n,
        "n_images": len(tile_rows),
        "images_per_prompt": next(iter(repetitions)) if len(repetitions) == 1 else None,
        "standard_error": None,
        "ci_95": None,
    }
    if n > 1:
        from scipy.stats import t

        se = stdev(prompt_means) / sqrt(n)
        margin = float(t.ppf(0.975, df=n - 1)) * se
        metric["standard_error"] = se
        metric["ci_95"] = {"lower": value - margin, "upper": value + margin, "margin": margin}
    return metric


summary_relative = Path("results/benchmark") / run / "summary.json"
summary_path = WORK_DIR / summary_relative
scores_relative = summary_relative.with_name("clip_scores.csv")
scores_path = WORK_DIR / scores_relative

if image_zip:
    records = [
        {"image_path": str(image), "sample": samples[image.name], "status": "success", "composite": True}
        for image in images
    ]
    observations_path = OBSERVATIONS_JSONL
else:
    # Reuse the packaged benchmark's selection of composite and legacy records.
    read_records = """
from contextlib import closing
import json
from pathlib import Path
import sqlite3
import sys

from benchmark import COMPOSITE_INDEX, evaluation_records

with closing(sqlite3.connect(Path(sys.argv[1]).as_uri() + "?mode=ro", uri=True)) as db:
    records = list(evaluation_records(db, sys.argv[2]))

for row in records:
    if row["status"] == "success":
        row["image_path"] = str(Path(row["image_path"]).resolve())
        row["composite"] = row.get("image_index") == COMPOSITE_INDEX
print(json.dumps(records))
"""
    records = json.loads(subprocess.check_output(
        [sys.executable, "-c", read_records, str(db_path), CRITERION_NAME],
        cwd=WORK_DIR, text=True, encoding="utf-8",
    ))
    if not records:
        raise ValueError("Lượt chạy chưa có mẫu alignment.")
    observations_path = WORK_DIR / "observations.jsonl" if manifest["observations"] else None

observations = {}
if observations_path is not None:
    with observations_path.open(encoding="utf-8") as lines:
        for line in lines:
            record = json.loads(line)
            if str(record["criterion"]) != CRITERION_ID:
                continue
            key = prompt_key(record)
            if key in observations:
                raise ValueError(f"Observation trùng: {key}")
            observations[key] = record.get("human_alignment")

successful = [row for row in records if row["status"] == "success"]
prompt_keys = {prompt_key(row["sample"]) for row in successful}
human_scores = []
for key in prompt_keys:
    score = observations.get(key)
    if type(score) in (int, float) and score in range(1, 6):
        human_scores.append(score)

metrics = {}
summary_path.parent.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(dir=WORK_DIR, prefix="tiles-") as tile_directory:
    tile_rows = split_tiles(successful, Path(tile_directory)) if successful else []
    clip_scores = list(clip_scores_512(tile_rows)) if tile_rows else []
    metrics["clip_score"] = {
        **prompt_level_metric(tile_rows, clip_scores),
        "model": CLIP_MODEL_NAME,
        "input_image_size": [CLIP_INPUT_SIZE, CLIP_INPUT_SIZE],
    }

    with scores_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("image_filename", "dataset", "source_id", "tile_index", "clip_score"))
        for row, score in zip(tile_rows, clip_scores, strict=True):
            dataset, source_id = prompt_key(row["sample"])
            writer.writerow((row["image_filename"], dataset, source_id, row["tile_index"], score))

    try:
        multilingual_scores = list(multilingual_clip_scores(tile_rows)) if tile_rows else []
        metrics["multilingual_clip_score"] = prompt_level_metric(tile_rows, multilingual_scores)
    except (ImportError, OSError) as error:
        metrics["multilingual_clip_score"] = {
            "value": None, "n": 0, "expected": len(prompt_keys), "error": str(error),
        }

metrics["human_alignment"] = {
    "value": mean(human_scores) if human_scores else None,
    "n": len(human_scores),
    "expected": len(prompt_keys),
}
summary = {
    "run": run,
    "criteria": {
        CRITERION_NAME: {
            "samples": len(records),
            "by_status": dict(Counter(row["status"] for row in records)),
            "metrics": metrics,
        },
    },
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
                        encoding="utf-8")

archive_prefix = INPUT_ZIP.stem if image_zip else (
    INPUT_ZIP.name.removesuffix("-colab-input.zip") if INPUT_ZIP.name.endswith("-colab-input.zip") else run
)
output_zip = INPUT_ZIP.with_name(f"{archive_prefix}-{CRITERION_NAME}-colab-results.zip")
with NamedTemporaryFile(dir=output_zip.parent, suffix=".zip", delete=False) as temporary:
    temporary_path = Path(temporary.name)
try:
    with ZipFile(temporary_path, "w", compression=ZIP_DEFLATED) as archive:
        archive.write(summary_path, summary_relative.as_posix())
        archive.write(scores_path, scores_relative.as_posix())
    temporary_path.replace(output_zip)
finally:
    temporary_path.unlink(missing_ok=True)

rows = []
for name, criterion in summary["criteria"].items():
    for metric, data in criterion["metrics"].items():
        value = data.get("value")
        coverage = f"{data['n']}/{data['expected']}" if "expected" in data else str(data.get("n", "—"))
        score_text = f"{value:.6g}" if value is not None else "—"
        note = data.get("error", "")
        if (interval := data.get("ci_95")) is not None:
            score_text = f"{score_text} ± {interval['margin']:.6g}"
            note = (f"KTC 95% [{interval['lower']:.6g}, {interval['upper']:.6g}]"
                    f" · {data['n']:,} prompt · {data['n_images']:,} ảnh")
        cells = (name, metric, score_text, coverage, note)
        rows.append("<tr>" + "".join(f"<td>{escape(cell)}</td>" for cell in cells) + "</tr>")
counts = "; ".join(
    f"{name}: {criterion['samples']} mẫu · {criterion['by_status']}"
    for name, criterion in summary["criteria"].items()
)
display(HTML(
    f"<p>{escape(counts)}</p>"
    "<table><thead><tr><th>Tiêu chí</th><th>Metric</th><th>Điểm</th><th>Số mẫu</th><th>Ghi chú</th></tr></thead>"
    "<tbody>" + "".join(rows) + "</tbody></table>"
))
if INPUT_SOURCE == "drive":
    print(f"ZIP kết quả trên Google Drive: {output_zip}")
else:
    files.download(str(output_zip))